## Embeddings for the whole (filtered) DP2

Took about an hour on GPU with 8 Dask workers.

In [ ]:
import lsdb
dia_object = lsdb.open_catalog('/astro/store/shire/hats/dash/hats/v30_0_6/dia_object_collection', columns=["ra","dec","diaObjectId","diaSource"])

In [ ]:
flag_cols = [c for c in dia_object.meta["diaSource"].columns if 'flag' in c.lower()]
query_str = "not (" + " or ".join(f"diaSource.{c}" for c in flag_cols) + ")"
cat = dia_object.query(query_str).query("diaSource.len() > 100")

In [ ]:
import onnxruntime as ort
from light_curve.embed import ATCAT

def get_model(provider):
    """Instantiates the ATCAT model for inference"""
    return ATCAT.from_hf(
        output="last",
        band_groups={"u": 0, "g": 1, "r": 2, "i": 3, "z": 4, "y": 5},
        ort_session_kwargs={"providers": [provider], "sess_options": _session_options()},
    )

def _session_options():
    """ONNX Runtime will try to pin threads to specific CPU cores. 
    This helps fix the affinity warnings."""
    sess_options = ort.SessionOptions()
    sess_options.intra_op_num_threads = 1
    sess_options.inter_op_num_threads = 1
    return sess_options

def compute_embeddings(time, flux, flux_err, band, *, provider):
    """Calculate embeddings for a lightcurve"""
    model = get_model(provider)
    emb = model(time, flux, flux_err, band)
    return {"embeddings.value": emb.flatten()}

In [ ]:
import pandas as pd
import numpy as np

def _hack(df, *, provider):
    if len(df) == 0:
        df["embeddings.value"] = pd.Series([], dtype=np.float32)
        return df
    return df.map_rows(
        compute_embeddings, 
        columns=["diaSource.midpointMjdTai","diaSource.psfFlux","diaSource.psfFluxErr","diaSource.band"], 
        provider=provider,
        row_container="args",
        append_columns=True
    )

embeddings = cat.map_partitions(lambda df: _hack(df, provider="CUDAExecutionProvider"))
embeddings

In [ ]:
from dask.distributed import Client

with Client(n_workers=8):
    embeddings.write_catalog("/astro/store/shire/hats/dash/dp2_embeddings", overwrite=True, as_collection=False)

<img src="images/dask_progress.png" width=1000 />
<img src="images/gpu_usage.png" width=500 />

Let's check how many objects remain after filtering, and what their distribution in the sky is:

In [ ]:
dp2_embeddings = lsdb.open_catalog("/astro/store/shire/hats/dash/dp2_embeddings/dia_object_lc")
print(len(dp2_embeddings))
dp2_embeddings.plot_pixels()

Notebook 5. performs similarity search again but now with this whole set of embeddings.